创建collection

In [6]:
from pymilvus import MilvusClient
from rich import print as rp
from sqlalchemy.dialects.oracle import vector

# client = MilvusClient(host='localhost', port=19530)
client = MilvusClient("http://localhost:19530")
collection_name="docs"
db_name="rag_db"
client.use_database(db_name=db_name)
client.create_collection(
    collection_name=collection_name,
    dimension=1024,#必须和使用的嵌入向量模型的维度相同
    metric_type="COSINE",#向量夹角计算方式，余弦计算
 )


#查看元数据

In [4]:

metadata=client.describe_collection(collection_name=collection_name)
rp(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 469230150679540592,
    'consistency_level': 2,
    'consistency_level_name': 'Bounded',
    'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'schema_version': 0,
    'created_timestamp': 469230770864259089,
    'update_timestamp': 469230770864259089
}

准备原始数据

In [3]:

embedding_input= [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
     "Milvus 是一个适合 AI 应用的向量数据库。",
     "RAG 的核心是先检索相关知识，再让大模型生成答案。",
     "Docker Desktop 可以方便地在本地运行 Milvus Standalone"
]

生成嵌入向量

In [4]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings
from langchain_openai import  OpenAIEmbeddings
load_dotenv(override=True)
my_api_key = os.getenv("ALL_API_KEY")
my_base_url = os.getenv("BASE_URL")
# 通用初始化
openai_client = OpenAI(
    api_key=my_api_key,
    base_url=my_base_url,
)
response = openai_client.embeddings.create(
    model="qwen3.7-text-embedding",
    dimensions=1024,# 可选值：2560、2048、1536、1024、768、512、256
    input=embedding_input

)
print(response.data[0].embedding)
vector=response.data[0].embedding




[0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575, -0.023045703768730164, -0.022211644798517227, -0.03097568452358246, -0.014897584915161133, -0.009630179032683372, 0.00036249507684260607, -0.024405863136053085, 0.0064415051601827145, -0.0013946436811238527, -0.008180198259651661, 0.005206455942243338, 0.00915540661662817, -0.035492438822984695, -0.0747317299246788, 0.03546677529811859, -0.05779390409588814, -0.03287477418780327, 0.01597544737160206, -0.023777110502123833, 0.016437387093901634, -0.01133037731051445, 0.015410852618515491, -0.022981546819210052, 0.03716055676341057, -0.03546677529811859, -0.035980042070150375, 0.007891485467553139, -0.018657268956303596, 0.004940198268741369, -0.01568031869828701, 0.04847810044884682, -0.013267961330711842, 0.02597132883965969, -0.00598598038777709, 0.006008435972034931, 0.02889695204794407, 0.018349308520555496, 0.002717109164223075, -0.0008845842094160616, 0.0098162386566400

封装为可以插入的数据格式

In [35]:
data=[
        {
            "id":i,
            "vector":vector,
            "text":embedding_input[i],
            "source":"demo"
        }
        for i in range (len(embedding_input))
]

插入数据

In [36]:
insert_result=client.upsert(
    collection_name=collection_name,
    data=data,
)
print(insert_result)

{'upsert_count': 4, 'ids': [0, 1, 2, 3]}


手动flush 因为milvus数据库不会第一时间将数据落盘，如果要看到写入效果，需要手动flush，将数据写入磁盘

In [37]:
client.flush(collection_name=collection_name)

查看collection统计信息

In [38]:
stats=client.get_collection_stats(collection_name=collection_name)
print(stats)

{'row_count': 5}


DQL操作-扫描操作

In [42]:
iterator=client.query_iterator(
    collection_name=collection_name,
    filters="",
    output_fields=["id","vector","text","source"],
)
i=0
while True:
    rows=iterator.next()

    if not rows:
        break
    for row in rows:
        print(f"第{i+1}条")
        print(f"id:{row["id"]},vector:{row["vector"][:5]},text:{row["text"]},source:{row["source"]},")
        i+=1
iterator.close()

第1条
id:0,vector:[0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575],text:LangChain 是一个用于构建 LLM 应用的开发框架。,source:demo,
第2条
id:1,vector:[0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575],text:Milvus 是一个适合 AI 应用的向量数据库。,source:demo,
第3条
id:2,vector:[0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575],text:RAG 的核心是先检索相关知识，再让大模型生成答案。,source:demo,
第4条
id:3,vector:[0.03746851906180382, -0.01739976368844509, -0.022609427571296692, -0.014602456241846085, -0.053123172372579575],text:Docker Desktop 可以方便地在本地运行 Milvus Standalone,source:demo,


通过主键查询数据

In [ ]:
result_ids=client.get(
    collection_name=collection_name,
    ids=[0,1,2,3]

)
print(result_ids)

相似度检索

In [7]:
query="什么是向量数据库？"

response=openai_client.embeddings.create(
    model="qwen3.7-text-embedding",
    dimensions=1024,# 可选值：2560、2048、1536、1024、768、512、256
    input=query

)
query_vector=response.data[0].embedding
query_results=client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["id","text","source","distance"],
)
for result in query_results[0]:
    rp(result)

{
    'id': 0,
    'distance': 0.2931569218635559,
    'entity': {'id': 0, 'text': 'LangChain 是一个用于构建 LLM 应用的开发框架。', 'source': 'demo'}
}

{
    'id': 1,
    'distance': 0.2931569218635559,
    'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}
}

{
    'id': 2,
    'distance': 0.2931569218635559,
    'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}
}